In [2]:
import os
import shutil
import hashlib
from PIL import Image
import pandas as pd
from collections import Counter

## Data Gathering
Dataset dikumpulkan dari beberapa sumber dan digabungkan ke dalam satu struktur folder berdasarkan jenis tanaman dan penyakit

In [3]:
RAW_PATH = "data/raw"
COMBINED_PATH = "data/combined"

os.makedirs(COMBINED_PATH, exist_ok=True)

In [4]:
dataset_mapping = {
    "corn-leaf-disease": "corn",
    "corn-or-maize-leaf-disease-dataset": "corn",
    "mango-leaf-disease-dataset": "mango",
    "potato-leaf-disease-dataset": "potato",
    "tomato-disease-multiple-sources": "tomato",
    "tomato-leaves-dataset": "tomato"
}

Dataset dari berbagai sumber dipetakan ke dalam kategori tanaman: corn; mango; potato; tomato

In [5]:
def clean_name(name):
    return name.lower().replace(" ", "_").replace("-", "_")

In [6]:
corn_mapping = {
    "daun_sehat": "healthy",
    "healthy": "healthy",

    "karat_daun": "rust",
    "common_rust": "rust",

    "hawar_daun": "blight",
    "blight": "blight",

    "bercak_daun": "leaf_spot",
    "gray_leaf_spot": "leaf_spot"
}

In [7]:
def short_name(filename):
    name, ext = os.path.splitext(filename)
    return hashlib.md5(name.encode()).hexdigest() + ext.lower()

Dilakukan standardisasi nama folder dan file untuk menghindari inkonsistensi data

In [8]:
def process_folder(source_path, plant_name):
    file_count = 0
    error_count = 0

    mapping = corn_mapping if plant_name == "corn" else {}

    for disease in os.listdir(source_path):
        disease_path = os.path.join(source_path, disease)

        if not os.path.isdir(disease_path):
            continue

        clean_disease = clean_name(disease)
        final_disease = mapping.get(clean_disease, clean_disease)

        dest_folder = os.path.join(COMBINED_PATH, plant_name, final_disease)
        os.makedirs(dest_folder, exist_ok=True)

        for file in os.listdir(disease_path):
            src_file = os.path.join(disease_path, file)

            if not os.path.isfile(src_file):
                continue

            try:
                new_name = short_name(file)
                dst_file = os.path.join(dest_folder, new_name)

                # copy file
                if not os.path.exists(dst_file):
                    shutil.copy(src_file, dst_file)
                    file_count += 1

            except Exception as e:
                error_count += 1
                print(f"skip: {file}")

    return file_count, error_count

Menggabungkan seluruh dataset ke dalam folder combined dengan struktur: plant -> disease -> images

In [9]:
for dataset_name, plant_name in dataset_mapping.items():
    dataset_path = os.path.join(RAW_PATH, dataset_name)

    if not os.path.exists(dataset_path):
        continue

    subfolders = os.listdir(dataset_path)

    if "train" in subfolders or "valid" in subfolders:

        for split in ["train", "valid"]:
            split_path = os.path.join(dataset_path, split)

            if os.path.exists(split_path):
                files, errors = process_folder(split_path, plant_name)

    else:
        files, errors = process_folder(dataset_path, plant_name)

## Data Assessing
Dilakukan evaluasi terhadap kualitas dataset, meliputi:
- Deteksi data duplikat
- Identifikasi gambar rusak
- Pemeriksaan format file

In [10]:
file_paths = []

for plant in os.listdir(COMBINED_PATH):
    plant_path = os.path.join(COMBINED_PATH, plant)

    for disease in os.listdir(plant_path):
        disease_path = os.path.join(plant_path, disease)

        for img in os.listdir(disease_path):
            img_path = os.path.join(disease_path, img)

            file_paths.append((plant, disease, img_path))

In [11]:
def get_image_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

hashes = {}
duplicates = []

for plant, disease, path in file_paths:
    try:
        img_hash = get_image_hash(path)

        if img_hash in hashes:
            duplicates.append(path)
        else:
            hashes[img_hash] = path

    except:
        pass

print("Jumlah duplikat:", len(duplicates))

Jumlah duplikat: 1166


In [13]:
bad_images = []

for _, _, path in file_paths:
    try:
        Image.open(path).verify()
    except:
        bad_images.append(path)

print("Gambar rusak:", len(bad_images))

Gambar rusak: 1


In [14]:
exts = set()

for _, _, path in file_paths:
    exts.add(path.split('.')[-1])

print(exts)

{'jpg', 'png', 'jpeg'}


### Insight
- Ditemukan data duplikat yang kemungkinan berasal dari penggabungan dataset berbeda
- Terdapat beberapa gambar yang tidak valid (corrupt)
- Format file bervariasi (jpg, jpeg, png)

## Data Cleaning
Dilakukan pembersihan dataset dengan:
- Menghapus data duplikat
- Menghapus gambar rusak
- Analisis distribusi data

In [15]:
for path in duplicates:
    os.remove(path)

print("Duplikat terhapus:", len(duplicates))

Duplikat terhapus: 1166


In [16]:
for path in bad_images:
    os.remove(path)

print("Gambar rusak terhapus:", len(bad_images))

Gambar rusak terhapus: 1


In [20]:
file_paths = []

for plant in os.listdir(COMBINED_PATH):
    for disease in os.listdir(os.path.join(COMBINED_PATH, plant)):
        for img in os.listdir(os.path.join(COMBINED_PATH, plant, disease)):
            file_paths.append((plant, disease, os.path.join(COMBINED_PATH, plant, disease, img)))

In [21]:
print("Total gambar:", len(file_paths))

Total gambar: 46055


In [22]:
counter = Counter()

for plant, disease, path in file_paths:
    counter[(plant, disease)] += 1

df_count = pd.DataFrame(
    [(p, d, c) for (p, d), c in counter.items()],
    columns=["plant", "disease", "count"]
)

df_count

,plant,disease,count
0,corn,blight,2146
1,corn,healthy,2162
2,corn,leaf_spot,998
3,corn,rust,2300
4,mango,anthracnose,486
5,mango,bacterial_canker,500
6,mango,cutting_weevil,500
7,mango,die_back,493
8,mango,gall_midge,500
9,mango,healthy,500


### Insight
- Data duplikat berhasil dihapus sehingga mengurangi potensi bias
- Gambar corrupt dihapus untuk menjaga kualitas dataset
- Distribusi data belum sepenuhnya seimbang, terutama pada dataset potato